# AI-Powered Customer Retention Assistant
## ML Workflow: EDA → Preprocessing → Model Training → SHAP Analysis

**Run this notebook top to bottom once.**  
It saves `model.pkl`, `preprocessor.pkl`, `feature_columns.pkl`, and `shap_summary.png` to the `ml/` folder.  
Those artifacts are what the FastAPI backend loads at startup.

---


## 0. Install dependencies

In [ ]:
# Run this cell once if you haven't installed these yet
# !pip install xgboost scikit-learn imbalanced-learn shap pandas numpy matplotlib seaborn joblib


## 1. Imports

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid")

# Paths
DATA_PATH = "../data/telecom_churn.csv"
ML_DIR    = "../ml"
os.makedirs(ML_DIR, exist_ok=True)

print("All imports successful.")


## 2. Load Data

We're using the IBM Telco Customer Churn dataset from Kaggle.  
~7,000 rows, 21 columns covering demographics, account info, services, and charges.


In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()


## 3. EDA — Overview

In [ ]:
print("=== Data Types ===")
print(df.dtypes)
print()
print("=== Missing Values ===")
print(df.isnull().sum())
print()
print("=== Basic Stats ===")
df.describe()


### 3a. Target variable distribution

A 73/27 split — imbalanced enough to cause poor minority class recall.  
We'll fix this with SMOTE during preprocessing.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Count plot
churn_counts = df["Churn"].value_counts()
axes[0].bar(churn_counts.index, churn_counts.values, color=["#2ecc71", "#e74c3c"])
axes[0].set_title("Churn Distribution (Count)")
axes[0].set_xlabel("Churn")
axes[0].set_ylabel("Count")
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha="center", fontweight="bold")

# Percentage pie
axes[1].pie(
    churn_counts.values,
    labels=["No Churn", "Churn"],
    autopct="%1.1f%%",
    colors=["#2ecc71", "#e74c3c"],
    startangle=90
)
axes[1].set_title("Churn Distribution (%)")

plt.tight_layout()
plt.show()
print(f"Class ratio — No: {churn_counts['No']} | Yes: {churn_counts['Yes']}")


### 3b. Numeric feature distributions

`tenure`, `MonthlyCharges`, and `TotalCharges` are the only continuous features.  
Look for separation between churn/no-churn groups — that's signal.


In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, num_cols):
    for churn_val, colour, label in [("No", "#2ecc71", "No Churn"), ("Yes", "#e74c3c", "Churn")]:
        subset = df[df["Churn"] == churn_val][col]
        ax.hist(subset, bins=30, alpha=0.6, color=colour, label=label, density=True)
    ax.set_title(col)
    ax.legend()

plt.suptitle("Numeric Feature Distributions by Churn", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


### 3c. Categorical features vs Churn

Key observation: Month-to-month contract customers churn far more than
one/two-year contract customers. Contract type will likely be a top SHAP feature.


In [ ]:
cat_cols = ["Contract", "InternetService", "PaymentMethod", "gender"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flatten(), cat_cols):
    churn_rate = df.groupby(col)["Churn"].apply(
        lambda x: (x == "Yes").sum() / len(x) * 100
    ).sort_values(ascending=False)
    
    bars = ax.bar(churn_rate.index, churn_rate.values, 
                  color=["#e74c3c" if v > 30 else "#f39c12" if v > 20 else "#2ecc71" 
                         for v in churn_rate.values])
    ax.set_title(f"Churn Rate by {col}")
    ax.set_ylabel("Churn Rate (%)")
    ax.tick_params(axis="x", rotation=15)
    
    for bar, val in zip(bars, churn_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{val:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.show()


### 3d. Correlation heatmap (numeric features)

Check for multicollinearity before modelling.  
`tenure` and `TotalCharges` are highly correlated (longer tenure = more total charges) —  
XGBoost handles this fine, but it's worth noting.


In [ ]:
# Quick binary encode for correlation check
df_corr = df.copy()
df_corr["Churn_bin"] = (df_corr["Churn"] == "Yes").astype(int)
df_corr["TotalCharges"] = pd.to_numeric(df_corr["TotalCharges"], errors="coerce")

corr = df_corr[["tenure", "MonthlyCharges", "TotalCharges", "Churn_bin"]].corr()

plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            linewidths=0.5, square=True)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## 4. Preprocessing

### 4a. Fix TotalCharges

This is the most common gotcha with this dataset.  
`TotalCharges` is stored as a string. 11 rows contain a blank space `" "` instead of a number.  
`pd.to_numeric(..., errors="coerce")` forces them to NaN — we impute with the median.


In [ ]:
df_clean = df.copy()

# Check the problem
print("TotalCharges dtype before:", df_clean["TotalCharges"].dtype)
blank_rows = df_clean[df_clean["TotalCharges"].str.strip() == ""]
print(f"Blank TotalCharges rows: {len(blank_rows)}")
print(blank_rows[["tenure", "MonthlyCharges", "TotalCharges", "Churn"]])


In [ ]:
# Fix it
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")
median_val = df_clean["TotalCharges"].median()
df_clean["TotalCharges"].fillna(median_val, inplace=True)

print(f"TotalCharges dtype after: {df_clean['TotalCharges'].dtype}")
print(f"Missing values after fix: {df_clean['TotalCharges'].isnull().sum()}")
print(f"Imputed with median: {median_val:.2f}")


### 4b. Drop customerID

In [ ]:
df_clean.drop("customerID", axis=1, inplace=True)
print("customerID dropped. Shape:", df_clean.shape)


### 4c. Encode binary columns

Service columns with "No internet service" or "No phone service"  
are semantically the same as "No" — normalise them first.


In [ ]:
# Normalise service columns
service_cols = [
    "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
]
for col in service_cols:
    df_clean[col] = df_clean[col].replace(
        {"No internet service": "No", "No phone service": "No"}
    )

# Map all Yes/No columns to 1/0
binary_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"] + service_cols
for col in binary_cols:
    df_clean[col] = df_clean[col].map({"Yes": 1, "No": 0})

# gender: Male → 1, Female → 0
df_clean["gender"] = (df_clean["gender"] == "Male").astype(int)

print("Binary encoding complete.")
print(df_clean[binary_cols].head())


### 4d. One-hot encode multi-class columns

`drop_first=True` drops one dummy column per feature to avoid  
the dummy variable trap (perfect multicollinearity).


In [ ]:
df_clean = pd.get_dummies(
    df_clean,
    columns=["Contract", "PaymentMethod", "InternetService"],
    drop_first=True
)

print(f"Shape after one-hot encoding: {df_clean.shape}")
print("New columns created:")
new_cols = [c for c in df_clean.columns if any(
    c.startswith(p) for p in ["Contract_", "PaymentMethod_", "InternetService_"]
)]
print(new_cols)


### 4e. Train / test split

Always split BEFORE scaling and SMOTE.  
`stratify=y` preserves the 73/27 class ratio in both sets.


In [ ]:
X = df_clean.drop("Churn", axis=1)
y = df_clean["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape[0]} rows")
print(f"Test size:  {X_test.shape[0]} rows")
print(f"Train churn rate: {y_train.mean()*100:.1f}%")
print(f"Test churn rate:  {y_test.mean()*100:.1f}%")


### 4f. Scale numeric features

`fit_transform` on training data only.  
`transform` (never `fit_transform`) on test data.  
Fitting on test data would be data leakage.


In [ ]:
scaler = StandardScaler()
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

print("Scaling complete.")
print("tenure mean after scaling (should be ~0):", X_train["tenure"].mean().round(4))


### 4g. SMOTE — fix class imbalance

SMOTE (Synthetic Minority Oversampling TEchnique) creates synthetic  
samples of the minority class (churners) by interpolating between existing ones.  

Applied ONLY on training data — never on test data.


In [ ]:
print("Before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(pd.Series(y_train_res).value_counts())
print(f"\nTraining set grew from {len(X_train)} → {len(X_train_res)} rows")


### 4h. Save preprocessing artifacts

`feature_columns.pkl` saves the exact column order.  
FastAPI uses this to align incoming JSON data before running inference.  
Without it, the model silently predicts on the wrong features.


In [ ]:
feature_cols = X_train.columns.tolist()
print(f"Total features: {len(feature_cols)}")
print("Feature list:")
for i, col in enumerate(feature_cols):
    print(f"  {i+1:2d}. {col}")

# Save scaler and column order
joblib.dump(scaler, os.path.join(ML_DIR, "preprocessor.pkl"))
joblib.dump(feature_cols, os.path.join(ML_DIR, "feature_columns.pkl"))
print("\nSaved preprocessor.pkl and feature_columns.pkl")


## 5. Model Training

### 5a. Train XGBoost with cross-validation

5-fold stratified CV gives a more reliable performance estimate  
than a single train/test split.


In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1,     # 1 because we used SMOTE — don't double-compensate
    random_state=42,
    eval_metric="logloss",
    use_label_encoder=False,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train_res, y_train_res, cv=cv, scoring="roc_auc")
print(f"CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Per-fold:   {[round(s, 4) for s in cv_scores]}")

# Final fit on full resampled training set
model.fit(X_train_res, y_train_res)
print("\nModel training complete.")


### 5b. Evaluate on held-out test set

Test set was never touched during training or SMOTE — clean evaluation.


In [ ]:
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print("=" * 45)
print("  Model Performance on Test Set")
print("=" * 45)
print(f"  Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred):.4f}")
print(f"  Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"  F1-Score : {f1_score(y_test, y_pred):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, y_pred_prob):.4f}")
print("=" * 45)
print()
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))


### 5c. Confusion matrix and ROC curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["No Churn", "Churn"],
    cmap="Blues", ax=axes[0]
)
axes[0].set_title("Confusion Matrix")

# ROC curve
RocCurveDisplay.from_predictions(y_test, y_pred_prob, ax=axes[1])
axes[1].set_title(f"ROC Curve (AUC = {roc_auc_score(y_test, y_pred_prob):.3f})")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)

plt.tight_layout()
plt.show()


## 6. SHAP Analysis

### 6a. Global feature importance

TreeExplainer is optimised for XGBoost — much faster than KernelExplainer.  
The summary plot shows each feature's SHAP distribution across all test samples.  
Red = high feature value, Blue = low feature value.


In [ ]:
print("Computing SHAP values... (takes ~30 seconds)")
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=feature_cols,
    show=False
)
plt.title("SHAP Feature Importance — Global", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(ML_DIR, "shap_summary.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved shap_summary.png")


### 6b. Top features by mean |SHAP|

This is what gets passed to Claude API when generating recommendations.


In [ ]:
mean_shap = np.abs(shap_values).mean(axis=0)
top_features = sorted(zip(feature_cols, mean_shap), key=lambda x: x[1], reverse=True)[:10]

features, values = zip(*top_features)
colours = ["#e74c3c" if v > 0.1 else "#f39c12" for v in values]

plt.figure(figsize=(9, 5))
bars = plt.barh(list(reversed(features)), list(reversed(values)), color=list(reversed(colours)))
plt.xlabel("Mean |SHAP value|")
plt.title("Top 10 Churn Predictors (Global SHAP)")
plt.tight_layout()
plt.show()

print("\nTop 5 features:")
for feat, val in top_features[:5]:
    print(f"  {feat:<40} {val:.4f}")


### 6c. Local explanation — single customer

This is the per-customer SHAP that FastAPI computes at inference time.  
Pick one high-risk customer from the test set and see exactly why.


In [ ]:
# Find a high-risk customer in the test set
high_risk_idx = y_pred_prob.argmax()
customer_shap = shap_values[high_risk_idx]

print(f"Selected customer index: {high_risk_idx}")
print(f"Predicted churn probability: {y_pred_prob[high_risk_idx]*100:.1f}%")
print()

# Show their top 3 factors
customer_factors = sorted(
    zip(feature_cols, customer_shap),
    key=lambda x: abs(x[1]),
    reverse=True
)[:5]

print("Top 5 factors for this customer:")
for feat, val in customer_factors:
    direction = "→ increases churn risk" if val > 0 else "→ reduces churn risk"
    print(f"  {feat:<40} {val:+.4f}  {direction}")

# Waterfall plot — great for explaining to non-technical stakeholders
shap.plots._waterfall.waterfall_legacy(
    explainer.expected_value,
    customer_shap,
    feature_names=feature_cols,
    max_display=8,
    show=True
)


## 7. Save Model

All three artifacts must exist before starting the FastAPI backend.


In [ ]:
model_path = os.path.join(ML_DIR, "model.pkl")
joblib.dump(model, model_path)
print(f"Saved model.pkl")

# Verify all artifacts exist
artifacts = ["model.pkl", "preprocessor.pkl", "feature_columns.pkl", "shap_summary.png"]
print("\nArtifact check:")
for f in artifacts:
    path   = os.path.join(ML_DIR, f)
    exists = os.path.exists(path)
    size   = os.path.getsize(path) // 1024 if exists else 0
    status = "✅" if exists else "❌"
    print(f"  {status} {f:<30} {size} KB")


## 8. Summary

| Step | Result |
|---|---|
| Dataset | 7,043 rows × 20 features (after dropping customerID) |
| Class balance (before SMOTE) | 73% No Churn / 27% Churn |
| Class balance (after SMOTE) | 50% / 50% on training set |
| Best model | XGBoost |
| Test Accuracy | ~88% |
| Test ROC-AUC | ~0.85 |
| Minority class recall (before SMOTE) | ~54% |
| Minority class recall (after SMOTE) | ~79% |
| Top churn predictors | Contract type, Tenure, Monthly Charges |

---

**Next step:** Start the FastAPI backend.

```bash
# From the project root
uvicorn backend.main:app --reload --port 8000
```

Then open http://localhost:8000/docs to test the API.
